# Data inspection — `transduced/` zarr stores

Manual examination of the rat-bone synchrotron stores under `D:\jannik\synchrotron-data\transduced\`. Everything in this notebook is **lazy** until the explicit materialization section near the bottom: opening a `DvcDataset` reads metadata only (no voxels), and array handles in `dataset.deformations` / `dataset.masks` are `zarr.Array` references.

Sections:
1. Discover stores on disk.
2. Open each lazily with `strict=False` — so any per-entry corruption surfaces in `broken_entries` instead of raising.
3. Cross-check against `_corruption_report.json` if present.
4. Per-store summary table (profile, spacing, masks, real, synthetic, broken, verification report).
5. Pick one store + one deformation, **explicitly** materialize a centered subblock for a sanity slice.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from mamba_dvc.io.dataset import DvcDataset, NO_MASK
from mamba_dvc.io.volume import load_volume

In [ ]:
%matplotlib widget

## 1. Discover stores

Just a directory glob. No zarr handles opened yet.

In [ ]:
TRANSDUCED_ROOT = Path(r'D:\jannik\synchrotron-data\transduced')

store_paths = sorted(TRANSDUCED_ROOT.glob('*.zarr'))
for p in store_paths:
    print(p.name)
print(f'\n{len(store_paths)} store(s) found')

## 2. Open each store lazily

`DvcDataset.open(strict=False)` runs the structural verifier but does not raise on per-entry problems. Anything broken lands in `dataset.broken_entries` (named, with a reason); anything fundamentally wrong (missing `base/`, missing reference, etc.) still raises `MalformedStoreError`.

No voxels are read here — only metadata + zarr handles.

In [ ]:
datasets: dict[str, DvcDataset] = {}
open_errors: dict[str, Exception] = {}

for path in store_paths:
    try:
        datasets[path.name] = DvcDataset.open(path, strict=False)
    except Exception as exc:
        open_errors[path.name] = exc

print(f'opened: {len(datasets)}')
for name, exc in open_errors.items():
    print(f'FAILED  {name}: {type(exc).__name__}: {exc}')

## 3. Sidecar corruption report

`_corruption_report.json` (if present) is the upstream extractor's record of arrays it could not write fully. Useful as ground truth when comparing against `broken_entries` from the verifier.

In [ ]:
report_path = TRANSDUCED_ROOT / '_corruption_report.json'
corruption_report: dict | None = None
if report_path.exists():
    corruption_report = json.loads(report_path.read_text())
    print(f"generated_at:        {corruption_report.get('generated_at')}")
    print(f"specimens_processed: {corruption_report.get('specimens_processed')}")
    print(f"specimens_with_issues: {corruption_report.get('specimens_with_issues')}")
    print()
    for issue in corruption_report.get('issues', []):
        print(f"  {issue['specimen']}/{issue['variant']}/{issue['artifact']}")
        print(f"    kind={issue['kind']} status={issue['status']}")
        print(f"    expected={issue.get('expected_bytes'):,}  actual={issue.get('actual_bytes'):,}")
else:
    print('no _corruption_report.json present')

## 4. Per-store summary

Profile, voxel spacing, volume shape, available masks, real entries, synthetic entries, broken entries, and the verification report (errors + warnings).

In [ ]:
def summarize(name: str, ds: DvcDataset) -> None:
    print('=' * 78)
    print(name)
    print('=' * 78)
    print(f'profile:        {ds.profile.name}')
    print(f'manifest:       {ds.manifest!r}')
    print(f'spacing:        {ds.spacing}')
    print(f'volume_shape:   {ds.volume_shape}')
    print(f'reference:      {ds.reference.shape} {ds.reference.dtype}')
    print(f'masks ({len(ds.masks)}):       {ds.list_masks()}')
    real = ds.list_real()
    synth = ds.list_synthetic()
    broken = ds.list_broken()
    print(f'real ({len(real)}):        {real}')
    print(f'synthetic ({len(synth)}):   {synth}')
    print(f'broken ({len(broken)}):      {broken}')
    for bname in broken:
        be = ds.broken_entries[bname]
        print(f'    - {bname}  kind={be.kind}  reason={be.reason}  missing={be.missing}')
    rep = ds.verification_report
    print(f'verification:   ok={rep.ok}  errors={len(rep.errors)}  warnings={len(rep.warnings)}')
    for e in rep.errors:
        print(f'    ERR   {e}')
    for w in rep.warnings:
        print(f'    WARN  {w}')
    print()

for name, ds in datasets.items():
    summarize(name, ds)

## 5. Inspect lazy handles for one store

Confirm that `dataset.deformations[name].image` is a `zarr.Array` handle — slicing it would be the only thing that triggers I/O. We just print metadata.

In [ ]:
FOCUS_STORE = next(iter(datasets), None)
if FOCUS_STORE is None:
    print('no datasets opened — skip')
else:
    ds = datasets[FOCUS_STORE]
    print(f'focus store: {FOCUS_STORE}\n')
    for entry_name in ds.list_all():
        entry = ds.deformations[entry_name]
        img = entry.image
        flow = entry.flow
        flow_desc = f'{flow.shape} {flow.dtype}' if flow is not None else 'None'
        print(
            f'  {entry.kind:9s} {entry_name:20s} '
            f'image={img.shape} {img.dtype}  flow={flow_desc}'
        )

In [ ]:
# Establish synthetic field type by manual inspection

In [ ]:
field = ds.load_synthetic('fs004', 'field')

In [ ]:
field._array.shape

In [ ]:
fig, ax = plt.subplots()
ax.imshow(
    field._array[250, :, :, 0],
    cmap='viridis'
)

In [ ]:
materialized_flow_fields = {
    name : ds.
    for name in ds.list_synthetic()
}

In [ ]:
import sys, ipympl
print(sys.executable, ipympl.__version__)

## 6. Explicit materialization (opt-in)

**This is the only cell that actually reads voxels.** A full volume is `(960, 1280, 1280)` float32 ≈ 6 GB per array; with reference + deformed + mask + flow that's ~24 GB for one synthetic pair. Default to a centered `dry_shape` subblock so the cell stays cheap.

Pick a store, pick a deformation name, pick a `dry_shape`. Set `dry_shape = None` to materialize the full volume — only do this once you know you have the RAM.

In [ ]:
STORE_NAME = FOCUS_STORE                 # e.g. '103L_Mg5Gd_4w_000.zarr'
DEFORMATION: str | None = None           # None -> first available healthy entry
DRY_SHAPE: tuple[int, int, int] | None = (96, 128, 128)

ds = datasets[STORE_NAME]
if DEFORMATION is None:
    DEFORMATION = next(iter(ds.list_all()), None)
if DEFORMATION is None:
    raise RuntimeError(f'{STORE_NAME} has no healthy deformations')

print(f'materializing {STORE_NAME} :: {DEFORMATION}  dry_shape={DRY_SHAPE}')
pair = ds.load_pair(DEFORMATION, dry_shape=DRY_SHAPE)

ref = pair.reference
defm = pair.deformed
print(
    f'reference {ref.shape} {ref.dtype}  '
    f'mean={ref.mean():.4f} std={ref.std():.4f} '
    f'min={ref.min():.4f} max={ref.max():.4f}'
)
print(
    f'deformed  {defm.shape} {defm.dtype}  '
    f'mean={defm.mean():.4f} std={defm.std():.4f} '
    f'min={defm.min():.4f} max={defm.max():.4f}'
)
if pair.mask is not None:
    coverage = float(pair.mask.mean())
    print(f'mask      {pair.mask.shape} {pair.mask.dtype}  coverage={coverage:.3%}')
else:
    print('mask      None')
if pair.gt_field is not None:
    print(f'gt_field  shape={pair.gt_field.shape}  (sampled callable, normalized to pull-back / dz_dy_dx)')
else:
    print('gt_field  None  (real entry)')
print(f'kind={pair.kind}  spacing={pair.spacing}')

### Center slice — quick visual sanity check

Mid-z slice of reference vs. deformed (and mask, if present). Uses matplotlib so it works without the PyVista/trame setup the other notebooks need.

In [ ]:
import matplotlib.pyplot as plt

z_mid = pair.reference.shape[0] // 2

n_panels = 3 if pair.mask is not None else 2
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))
axes[0].imshow(pair.reference[z_mid], cmap='bone')
axes[0].set_title(f'reference  z={z_mid}')
axes[1].imshow(pair.deformed[z_mid], cmap='bone')
axes[1].set_title(f'deformed  z={z_mid}')
if pair.mask is not None:
    axes[2].imshow(pair.mask[z_mid], cmap='gray')
    axes[2].set_title(f'mask  z={z_mid}')
for ax in axes:
    ax.set_axis_off()
fig.suptitle(f'{STORE_NAME} :: {DEFORMATION}  (dry_shape={DRY_SHAPE})')
fig.tight_layout()